# Real Estate Feasibility Engine – Python App
## Step 4: Cash flow & returns (costs, landowner payments, P&L, IRR, NPV, peak investment, payback)

**What this notebook does**
1. Adds `engine/costs.py` – every cost head, the land section and the calculators (BAR, AIF, initial investment).
2. Adds `engine/cashflow.py` – quarterly outflows, landowner payments with advance recovery, project & developer cash flow, the P&L and all 18 key metrics, plus `run_model()` which runs the **whole engine in one call**.
3. Shows the results like the Excel OUTPUT and CASHFLOW sheets, with a funding-curve chart.
4. **Compares everything with your Excel FINAL file.**

**Before you start:** Steps 1–3 must have been run once. Run cells top to bottom with **Shift + Enter**.

### Cell 1 – Connect to Google Drive and open the project folder

In [ ]:
import os
try:
    from google.colab import drive
    drive.mount('/content/drive')
    BASE = '/content/drive/MyDrive/feasibility_app'
except ImportError:
    BASE = os.path.abspath('feasibility_app')
os.chdir(BASE)
for f in ['inputs.py', 'timeline.py', 'curves.py', 'collections.py']:
    assert os.path.exists(f'engine/{f}'), f'engine/{f} missing – please run the earlier steps first.'
print('Working in:', BASE)

### Cell 2 – Write the cost engine (`engine/costs.py`)
Exactly the Excel formulas, for example:
* **Land (JV)** = landowner share × total revenue · **Stamp duty (JV)** = land area × DLC rate × 2.75%
* **Construction heads** = sum of each category's products (villa, apartment, studio …)
* **Marketing** = 7% × built-up revenue + 5% × plotted revenue · **Legal** = ₹75 × saleable area
* **BAR** = extra BAR area × reserved price × authority % · **AIF** = principal × [(1 + r/m)^(m×t) − 1]
* **Initial investment (JV)** = landowner advance (token + DD + DA) + legal share + stamp duty + brokerage + BAR share + pre-DA items

In [ ]:
%%writefile engine/costs.py
"""
engine/costs.py  -  STEP 4a: HOW MUCH does each cost head cost?  (Excel INPUTS Sections 2, 6 and 7)

  * land section: land value, land cost / landowner share, stamp duty, brokerage base
  * all 21 cost heads (+ spare rows) with the exact Excel calculation for each
  * calculators: BAR / premium FAR, AIF interest, initial investment (JV and Outright)
All amounts in Rs Crore.
"""
from __future__ import annotations

from dataclasses import dataclass, field

from .inputs import Project, CRORE, SQM_PER_SQYD, SQFT_PER_SQYD, CONSTRUCTION_HEADS, COMPOUNDING
from .curves import Curves


@dataclass
class CostResult:
    total_sa: float = 0.0                 # total saleable area of active products (sq ft)
    total_revenue: float = 0.0            # GDV (Rs Cr)
    total_construction: float = 0.0       # sum of product construction cost (Rs Cr)
    plotted_revenue: float = 0.0
    land_value_market: float = 0.0
    land_value_purchase: float = 0.0
    land_cost: float = 0.0                # JV: landowner revenue share | Outright: purchase price
    stamp_duty: float = 0.0
    brokerage_base: float = 0.0
    dev_share: float = 1.0
    amounts: dict = field(default_factory=dict)        # cost key -> Rs Cr
    total_cost: float = 0.0
    construction_heads_total: float = 0.0
    bar: dict = field(default_factory=dict)            # BAR calculator details
    aif: dict = field(default_factory=dict)
    initial_items: dict = field(default_factory=dict)  # JV initial-investment items (Rs Cr)
    initial_jv: float = 0.0
    initial_outright: float = 0.0
    initial_used: float = 0.0
    checks: list = field(default_factory=list)


def build_costs(project: Project, cv: Curves) -> CostResult:
    r = CostResult()
    L, info = project.land, project.info
    jv = info.deal_type == "JV"
    active = project.active_products()

    r.total_sa = sum(p.area_sqft for p in active)
    r.total_revenue = sum(cv.revenue.values())
    r.total_construction = sum(cv.construction_cost.values())
    r.plotted_revenue = sum(cv.revenue[p.name] for p in active if p.zone == "Plotted Development")
    rev_cat = lambda cat: sum(cv.revenue[p.name] for p in active if p.category == cat)
    con_cat = lambda cat: sum(cv.construction_cost[p.name] for p in active if p.category == cat)

    # ---------------- land section
    dlc_sqyd = L.dlc_rate_sqm * SQM_PER_SQYD
    r.land_value_market = L.area_sqyd * L.market_rate / CRORE
    r.land_value_purchase = L.area_sqyd * L.purchase_rate / CRORE
    r.dev_share = (1 - L.lo_share) if jv else 1.0
    r.land_cost = L.lo_share * r.total_revenue if jv else r.land_value_purchase
    r.stamp_duty = (L.area_sqyd * dlc_sqyd * L.stamp_duty_jv if jv
                    else L.area_sqyd * max(dlc_sqyd, L.purchase_rate) * L.stamp_duty_outright) / CRORE
    r.brokerage_base = r.land_value_market if jv else r.land_value_purchase

    # ---------------- BAR / premium FAR calculator
    b = project.bar
    free = L.area_sqyd * SQM_PER_SQYD * b.free_far
    planned = r.total_sa * (SQM_PER_SQYD / SQFT_PER_SQYD) / b.loading_factor if b.loading_factor > 0 else 0.0
    extra = max(0.0, planned - free)
    pct = (b.pct_table.get(b.land_use, {}) or {}).get(b.authority) or 0.0
    bar_cost = extra * b.reserved_price_sqm * pct / CRORE
    r.bar = dict(free_sqm=free, planned_sqm=planned, extra_sqm=extra, pct=pct, cost_excl_gst=bar_cost,
                 cost=bar_cost * (1 + (b.gst_rate if b.gst_applicable else 0.0)))

    # ---------------- cost heads (same formulas as Excel Section 6)
    A = r.amounts
    cat_of = {k: cat for k, _, cat in CONSTRUCTION_HEADS}
    gate = lambda c, v: v if c.applicable else 0.0
    order = [c.key for c in project.costs]
    for c in project.costs:                                  # first pass: everything except contingency & AIF
        k = c.key
        if k == "land":    A[k] = r.land_cost
        elif k == "sd":    A[k] = r.stamp_duty
        elif k == "brok":  A[k] = gate(c, c.rate1 * r.brokerage_base)
        elif k in cat_of:  A[k] = gate(c, con_cat(cat_of[k]))
        elif k == "ovh":   A[k] = gate(c, c.rate1 * r.total_revenue)
        elif k == "mkt":   A[k] = gate(c, c.rate1 * (r.total_revenue - r.plotted_revenue) + c.rate2 * r.plotted_revenue)
        elif k == "legal": A[k] = gate(c, c.rate1 * r.total_sa / CRORE)
        elif k == "ews":   A[k] = gate(c, c.rate1 * c.rate2 * r.total_sa / CRORE)
        elif k == "edc":
            v = {"Rs/sq ft (total saleable area)": c.rate1 * r.total_sa / CRORE,
                 "% of construction cost": c.rate1 * r.total_construction, "Lump sum (Rs Cr)": c.rate1}.get(c.rate2, 0.0)
            A[k] = gate(c, v)
        elif k == "hotel": A[k] = gate(c, c.rate1 * c.rate2 / CRORE)
        elif k == "furn":  A[k] = gate(c, c.rate1 * sum(p.area_sqft for p in active if p.category == "Studio") / CRORE)
        elif k == "gst":   A[k] = gate(c, (rev_cat("Commercial") + rev_cat("Plot")) * c.rate2 * c.rate1 * r.dev_share)
        elif k == "bar":   A[k] = gate(c, r.bar["cost"])
        elif k.startswith("spare"):
            v = {"Rs/sq ft (total saleable area)": (c.rate1 or 0) * r.total_sa / CRORE,
                 "% of total revenue": (c.rate1 or 0) * r.total_revenue,
                 "% of construction cost": (c.rate1 or 0) * r.total_construction,
                 "Lump sum (Rs Cr)": c.rate1 or 0,
                 "Rs/sq yd (land area)": (c.rate1 or 0) * L.area_sqyd / CRORE}.get(c.basis, 0.0)
            A[k] = gate(c, v)
    # AIF: interest on (land + stamp duty + brokerage)
    a = project.aif
    principal = A["land"] + A["sd"] + A.get("brok", 0.0)
    m = COMPOUNDING.get(a.compounding, 1)
    aif_cost = principal * ((1 + a.rate / m) ** (m * a.tenure_years) - 1)
    r.aif = dict(principal=principal, periods_per_year=m, cost=aif_cost)
    A["aif"] = gate(project.cost("aif"), aif_cost)
    # contingency: % of construction cost OR of all costs except land, stamp duty, brokerage and itself
    c = project.cost("cont")
    base_all = sum(v for k, v in A.items() if k not in ("land", "sd", "brok", "cont"))
    A["cont"] = gate(c, c.rate1 * (r.total_construction if c.rate2 == "Construction cost" else base_all))

    r.amounts = {k: A[k] for k in order}                    # keep the Excel order
    r.total_cost = sum(r.amounts.values())
    r.construction_heads_total = sum(A[k] for k in cat_of)

    # ---------------- initial investment
    ii = project.initial
    advance = ii.advance_pct * r.land_value_market
    r.initial_items = {
        "token": advance - ii.dd_amount - ii.da_amount, "dd": ii.dd_amount, "da": ii.da_amount,
        "legal": A["legal"] * ii.legal_share, "dev": ii.dev_cost_pre_da, "marketing": ii.marketing_pre_launch,
        "sd": A["sd"], "brok": A["brok"], "bar": A["bar"] * ii.bar_share}
    r.initial_jv = sum(r.initial_items.values())
    r.initial_outright = (A["land"] + A["sd"] + A["brok"]
                          + ii.outright_construction_share * (A["villa"] + A["apt"] + A["studio"])
                          + ii.outright_legal_share * A["legal"])
    r.initial_used = r.initial_jv if jv else r.initial_outright

    # ---------------- checks
    if abs(r.construction_heads_total - r.total_construction) > 1e-6 and all(project.cost(k).applicable for k in cat_of):
        r.checks.append("[Costs] construction heads do not add up to the product construction cost.")
    if r.initial_items["token"] < -1e-9:
        r.checks.append("[Costs] DD + DA instalments exceed the total landowner advance.")
    if project.cost("bar").applicable and pct == 0:
        r.checks.append(f"[Costs] no BAR % for {b.authority} / {b.land_use}.")
    return r


### Cell 3 – Write the cash-flow engine (`engine/cashflow.py`)
* **Outflow of each cost head** = amount × its quarterly % (from Step 2). In a JV, initial-investment items are paid at their payout quarter and **only the balance** follows the curve – nothing is counted twice.
* **Landowner:** the advance goes out first; then he receives nothing until his share of collections exceeds it → *cumulative paid = greater of (advance so far, share earned so far)*.
* **Project cash flow** = collections − costs (before landowner share). **Developer cash flow** = project cash flow − cash paid to the landowner.
* **IRR** = annualised from quarterly flows: (1 + quarterly IRR)⁴ − 1 · **NPV** at the cost of equity · **Peak investment** = deepest point of cumulative developer cash · **Payback** = quarter cumulative cash returns to zero.

In [ ]:
%%writefile engine/cashflow.py
"""
engine/cashflow.py  -  STEP 4b: CASH FLOW, P&L and RETURNS  (Excel CASHFLOW + OUTPUT sheets)

  * outflow of every cost head = amount x its quarterly % (from CURVES).
    In a JV the initial-investment items (stamp duty, brokerage, legal, BAR share, pre-DA development,
    pre-launch marketing) are paid early at their payout quarter and only the BALANCE follows the curve,
    so nothing is counted twice.
  * landowner (JV): the advance (token, DD, DA) is paid first; afterwards he receives nothing until his
    share of collections exceeds the advance  ->  cumulative paid = max(cumulative advance, cumulative share).
  * project cash flow (before landowner share) and developer cash flow (after it)
  * P&L, IRR, NPV, peak investment, break-even, payback, equity multiple, ROCE, CAGR, yearly (FY) summary
"""
from __future__ import annotations

from dataclasses import dataclass, field

import numpy as np

from .inputs import Project, HORIZON_Q, CONSTRUCTION_HEADS
from .timeline import Timeline
from .curves import Curves
from .collections import Collections
from .costs import CostResult

PULL = {"sd": "sd", "brok": "brok", "legal": "legal", "bar": "bar", "mkt": "marketing"}   # cost key -> initial item


# ------------------------------------------------------------------ IRR / NPV helpers
def npv(rate: float, flows: np.ndarray) -> float:
    """Excel NPV: first flow discounted one period."""
    t = np.arange(1, len(flows) + 1)
    return float(np.sum(flows / (1 + rate) ** t))


def irr(flows: np.ndarray):
    """Quarterly IRR by bisection (same root as Excel for normal 'invest first, earn later' flows).
    Returns None when the cash flow never changes sign."""
    flows = np.asarray(flows, dtype=float)
    if not (np.any(flows > 0) and np.any(flows < 0)):
        return None
    f = lambda r: np.sum(flows / (1 + r) ** np.arange(len(flows)))
    lo, hi = -0.99, 1.0
    while f(hi) > 0 and hi < 1e6:
        hi *= 2
    if f(lo) * f(hi) > 0:
        return None
    for _ in range(300):
        mid = (lo + hi) / 2
        if f(lo) * f(mid) <= 0: hi = mid
        else: lo = mid
    return (lo + hi) / 2


def _first_q(mask: np.ndarray):
    idx = np.flatnonzero(mask)
    return int(idx[0]) + 1 if idx.size else None


@dataclass
class CashFlow:
    inflow: np.ndarray = None
    outflows: dict = field(default_factory=dict)       # cost key -> 40 values (landowner share excluded)
    total_outflow: np.ndarray = None
    lo_advance: np.ndarray = None
    lo_paid: np.ndarray = None
    project_cf: np.ndarray = None
    developer_cf: np.ndarray = None
    tax: np.ndarray = None
    developer_cf_post_tax: np.ndarray = None
    pnl: dict = field(default_factory=dict)
    metrics: dict = field(default_factory=dict)
    fy_summary: dict = field(default_factory=dict)
    checks: list = field(default_factory=list)


def build_cashflow(project: Project, tl: Timeline, cv: Curves, co: Collections, cr: CostResult) -> CashFlow:
    cf = CashFlow()
    jv = project.info.deal_type == "JV"
    q_ = project.initial.payout_q
    A, items = cr.amounts, cr.initial_items
    cons_keys = {k for k, _, _ in CONSTRUCTION_HEADS}

    def at(q: int, amount: float) -> np.ndarray:
        v = np.zeros(HORIZON_Q)
        if 1 <= q <= HORIZON_Q: v[q - 1] = amount
        return v

    # ---------------- outflows by cost head
    cf.inflow = co.total.copy()
    for c in project.costs:
        k, G, pct = c.key, A[c.key], cv.cost_pct[c.key]
        if k == "land":
            cf.outflows[k] = np.zeros(HORIZON_Q) if jv else G * pct
        elif k in PULL:
            pa = items[PULL[k]] if jv else 0.0
            cf.outflows[k] = at(q_[PULL[k]], pa) + (G - pa) * pct
            if pa < -1e-9 or pa > G + 1e-9:
                cf.checks.append(f"[Cash flow] early payment for '{c.name}' exceeds its cost head.")
        elif k in cons_keys:
            pa = items["dev"] * G / cr.construction_heads_total if (jv and cr.construction_heads_total > 0) else 0.0
            cf.outflows[k] = at(q_["dev"], pa) + (G - pa) * pct
        else:
            cf.outflows[k] = G * pct
    cf.total_outflow = sum(cf.outflows.values())

    # ---------------- landowner: advance first, then recovered from his share
    if jv:
        cf.lo_advance = at(q_["token"], items["token"]) + at(q_["dd"], items["dd"]) + at(q_["da"], items["da"])
    else:
        cf.lo_advance = np.zeros(HORIZON_Q)
    paid_cum = np.maximum(np.cumsum(cf.lo_advance), np.cumsum(co.landowner))
    cf.lo_paid = np.diff(np.concatenate([[0.0], paid_cum]))

    # ---------------- net cash flows
    cf.project_cf = cf.inflow - cf.total_outflow
    cf.developer_cf = cf.project_cf - cf.lo_paid

    # ---------------- P&L (Excel OUTPUT sections A–D)
    rev, cost = cr.total_revenue, cr.total_cost
    ebitda = rev - cost
    tax_rate = project.finance.tax_llp if project.info.entity == "LLP" else project.finance.tax_company
    tax = max(0.0, ebitda) * tax_rate
    pat = ebitda - tax
    dev_rev = rev * cr.dev_share
    cf.pnl = dict(revenue=rev, total_cost=cost, ebitda=ebitda, ebitda_margin=ebitda / rev if rev else 0.0,
                  tax_rate=tax_rate, tax=tax, pat=pat, pat_margin=pat / rev if rev else 0.0,
                  developer_revenue=dev_rev, developer_cost=dev_rev - ebitda,
                  ebitda_margin_on_dev_share=ebitda / dev_rev if dev_rev else 0.0,
                  landowner_revenue=rev - dev_rev, construction_heads_total=cr.construction_heads_total)
    cf.tax = tax * co.profile
    cf.developer_cf_post_tax = cf.developer_cf - cf.tax

    # ---------------- returns & funding metrics (Excel CASHFLOW section E + OUTPUT section E)
    qrate = (1 + project.finance.cost_of_equity) ** 0.25 - 1
    annual = lambda q: None if q is None else (1 + q) ** 4 - 1
    p_irr_q, d_irr_q = irr(cf.project_cf), irr(cf.developer_cf)
    cum_d = np.cumsum(cf.developer_cf)
    peak = max(0.0, -cum_d.min())
    peak_q = int(np.argmin(cum_d)) + 1 if peak > 0 else None
    running_min = np.minimum.accumulate(cum_d)
    payback_q = _first_q((cum_d >= -1e-6) & (running_min < -1e-6))
    life = project.info.lifecycle_override_years or tl.last_q / 4
    ii = cr.initial_used
    land_heads = A["land"] + A["sd"] + A["brok"]
    em = (dev_rev / (cost - land_heads) if (cost - land_heads) else 0.0) if jv else (rev / cost if cost else 0.0)
    roce = ebitda / ii / life if ii > 0 and life > 0 else 0.0
    roi = pat + ii
    cagr = (roi / ii) ** (1 / life) - 1 if ii > 0 and life > 0 and roi > 0 else 0.0
    pos, neg = cf.developer_cf[cf.developer_cf > 0].sum(), -cf.developer_cf[cf.developer_cf < 0].sum()
    cf.metrics = dict(
        project_irr=annual(p_irr_q), developer_irr=annual(d_irr_q),
        developer_irr_post_tax=annual(irr(cf.developer_cf_post_tax)),
        project_npv=npv(qrate, cf.project_cf), developer_npv=npv(qrate, cf.developer_cf),
        peak_investment=peak, peak_quarter=peak_q, peak_date=tl.end_dates[peak_q - 1] if peak_q else None,
        largest_quarter_deficit=max(0.0, -cf.developer_cf.min()),
        breakeven_project_q=_first_q(cf.project_cf > 1e-6), breakeven_developer_q=_first_q(cf.developer_cf > 1e-6),
        payback_q=payback_q, payback_date=tl.end_dates[payback_q - 1] if payback_q else None,
        cash_multiple=pos / neg if neg else 0.0, equity_multiple=em, initial_investment=ii, lifecycle_years=life,
        roce=roce, return_on_investment=roi, cagr=cagr, pat_margin=cf.pnl["pat_margin"], ebitda_margin=cf.pnl["ebitda_margin"],
        land_cost_to_gdv=A["land"] / rev if rev else 0.0, initial_investment_to_gdv=ii / rev if rev else 0.0)

    # benchmark status (green = beats benchmark)
    bm = project.finance.benchmarks
    pairs = {"project_irr": "project_irr", "developer_irr": "developer_irr", "equity_multiple": "equity_multiple",
             "roce": "roce", "cagr": "cagr", "pat_margin": "pat_margin", "ebitda_margin": "ebitda_margin"}
    cf.metrics["status"] = {k: ("Strong" if cf.metrics[k] is not None and cf.metrics[k] > bm[b] else "Review")
                            for k, b in pairs.items()}

    # ---------------- yearly summary by Indian financial year
    for i, fy in enumerate(tl.fy):
        s = cf.fy_summary.setdefault(fy, dict(collections=0.0, costs_paid=0.0, paid_to_landowner=0.0, developer_net=0.0))
        s["collections"] += cf.inflow[i]; s["costs_paid"] += cf.total_outflow[i]
        s["paid_to_landowner"] += cf.lo_paid[i]; s["developer_net"] += cf.developer_cf[i]

    # ---------------- checks (same as Excel CASHFLOW section H)
    for c in project.costs:
        expected = 0.0 if (jv and c.key == "land") else A[c.key]
        if abs(cf.outflows[c.key].sum() - expected) > 1e-6:
            cf.checks.append(f"[Cash flow] '{c.name}' paid {cf.outflows[c.key].sum():.4f} instead of {expected:.4f}.")
    if jv and (abs(cf.lo_paid.sum() - co.landowner.sum()) > 1e-6):
        cf.checks.append("[Cash flow] landowner not paid exactly his share (advance not fully recovered).")
    if abs(cf.developer_cf.sum() - ebitda) > 1e-6:
        cf.checks.append("[Cash flow] developer total cash does not equal EBITDA.")
    if abs(cf.tax.sum() - tax) > 1e-6:
        cf.checks.append("[Cash flow] tax spread does not equal P&L tax.")
    return cf


def run_model(project: Project) -> dict:
    """One call that runs the whole engine in order and returns every result."""
    from .inputs import validate
    from .timeline import build_timeline
    from .curves import build_curves
    from .collections import build_collections
    from .costs import build_costs
    tl = build_timeline(project)
    cv = build_curves(project, tl)
    co = build_collections(project, tl, cv)
    cr = build_costs(project, cv)
    cf = build_cashflow(project, tl, cv, co, cr)
    problems = validate(project) + tl.errors + cv.checks + co.checks + cr.checks + cf.checks
    return dict(timeline=tl, curves=cv, collections=co, costs=cr, cashflow=cf, problems=problems)


### Cell 4 – Update the Excel comparison helper (adds the CASHFLOW / OUTPUT reader)

In [ ]:
%%writefile engine/excel_check.py
"""
engine/excel_check.py - reads the Excel FINAL model and compares it with the Python engine.
Works by LABELS (not cell addresses), so it is robust to row positions.
"""
from __future__ import annotations
import numpy as np
import openpyxl

NQ = 40
QCOL0 = 4      # quarter columns start at column D in TIMELINE / CURVES


def _row_values(ws, r):
    return np.array([ws.cell(r, c).value or 0 for c in range(QCOL0, QCOL0 + NQ)], dtype=float)


def read_excel_curves(path: str) -> dict:
    """Returns {'sales': {product: arr}, 'cons': {...}, 'cost': {cost name: arr}, 'profiles': {...}}"""
    wb = openpyxl.load_workbook(path, data_only=True, read_only=False)
    ws = wb["CURVES"]
    out = {"sales": {}, "cons": {}, "cost": {}, "cost_order": [], "profiles": {}}
    block, section = None, None
    for r in range(1, ws.max_row + 1):
        label = ws.cell(r, 2).value
        if not isinstance(label, str):
            continue
        if label.startswith("B.  PRODUCT"):   section = "products"; continue
        if label.startswith("C.  AGGREGATE"): section = "profiles"; continue
        if label.startswith("D.  COST HEAD"): section = "costs"; continue
        if label.startswith("E.  CURVE CHECKS"): break
        if section == "products":
            if label.startswith("   Sales % (curve"):         out["sales"][block] = _row_values(ws, r)
            elif label.startswith("   Construction % (curve"): out["cons"][block] = _row_values(ws, r)
            elif not label.startswith("   "):                   block = label
        elif section == "profiles":
            out["profiles"][label] = _row_values(ws, r)
        elif section == "costs":
            if label.startswith("   Spend %"): out["cost"][block] = _row_values(ws, r); out["cost_order"].append(block)
            elif not label.startswith("   "):   block = label
    return out


def read_excel_timeline(path: str) -> dict:
    wb = openpyxl.load_workbook(path, data_only=True)
    ws = wb["INPUTS"]
    prods = {}
    for r in range(1, ws.max_row + 1):          # product table: active rows have quarters in P..S
        if ws.cell(r, 5).value == "Yes" and isinstance(ws.cell(r, 16).value, (int, float)) and isinstance(ws.cell(r, 2).value, str):
            prods[ws.cell(r, 2).value] = tuple(int(ws.cell(r, c).value) for c in (16, 17, 18, 19))
    return prods


def read_excel_collections(path: str) -> dict:
    """Bookings, collections, totals, JV split and milestone due quarters from the COLLECTIONS sheet."""
    wb = openpyxl.load_workbook(path, data_only=True)
    ws = wb["COLLECTIONS"]
    out = {"bookings": {}, "collections": {}, "triggers": {}, "total": None, "landowner": None, "developer": None}
    section, block = None, None
    for r in range(1, ws.max_row + 1):
        label = ws.cell(r, 2).value
        if not isinstance(label, str):
            continue
        if label.startswith("A.  MILESTONE"): section = "A"; continue
        if label.startswith("B.  PRODUCT"):   section = "B"; continue
        if label.startswith("C.  PROJECT"):   section = "C"; continue
        if label.startswith("D.  COLLECTION"): break
        if section == "A":
            if "   →   " in label:
                block = label.split("   →   ")[0]; out["triggers"][block] = []
            elif block and label.strip() and not label.startswith("   ") and ws.cell(r, 5).value:
                due = ws.cell(r, 6).value
                out["triggers"][block].append((label, int(due) if isinstance(due, (int, float)) else None))
        elif section == "B":
            if label.startswith("   Bookings (sales"):        out["bookings"][block] = _row_values(ws, r)
            elif label.startswith("   Collections (cash"):    out["collections"][block] = _row_values(ws, r)
            elif not label.startswith("   "):                 block = label
        elif section == "C":
            if label.startswith("Total collections (cash"):   out["total"] = _row_values(ws, r)
            elif label.startswith("   Landowner share"):      out["landowner"] = _row_values(ws, r)
            elif label.startswith("   Developer share"):      out["developer"] = _row_values(ws, r)
    return out


def read_excel_cashflow(path: str) -> dict:
    """Quarterly cash rows from the CASHFLOW sheet and P&L / key metrics from the OUTPUT sheet."""
    wb = openpyxl.load_workbook(path, data_only=True)
    ws = wb["CASHFLOW"]
    out = {"outflows": [], "rows": {}, "pnl": {}, "costs": [], "metrics": {}}
    want = {"Customer collections (100%)": "inflow", "Total outflows (excl. landowner share)": "total_outflow",
            "Cash paid to landowner this quarter": "lo_paid", "Developer net cash flow (pre-tax)": "developer_cf",
            "Developer net cash flow (post-tax, indicative)": "developer_cf_post_tax"}
    section = None
    for r in range(1, ws.max_row + 1):
        label = ws.cell(r, 2).value
        if not isinstance(label, str):
            continue
        if label.startswith("B.  CASH OUTFLOWS"): section = "B"; continue
        if label.startswith("C.  LANDOWNER"): section = None
        if label.startswith("Project net cash flow"): out["rows"]["project_cf"] = _row_values(ws, r)
        if section == "B" and not label.startswith("Total outflows"):
            out["outflows"].append(_row_values(ws, r))
        if label in want: out["rows"][want[label]] = _row_values(ws, r)
        if label.startswith("Total outflows"): section = None
    ws = wb["OUTPUT"]
    section = None
    for r in range(1, ws.max_row + 1):
        title = ws.cell(r, 2).value
        if isinstance(title, str):
            if title.startswith("B.  COST STATEMENT"): section = "B"
            elif title.startswith("C.  PROFIT"): section = "C"
            elif title.startswith("D.  JV"): section = "D"
            elif title.startswith("E.  KEY"): section = "E"
            continue
        label, val = ws.cell(r, 3).value, ws.cell(r, 4).value
        if not isinstance(label, str):
            continue
        if section == "B" and label not in ("Cost head",) and not label.startswith("TOTAL") and not label.startswith("Total development"):
            out["costs"].append(val or 0.0)
        if label == "SUM TOTAL REVENUE (A)": out["pnl"]["revenue"] = val
        if label == "TOTAL COST (B)": out["pnl"]["total_cost"] = val
        if label == "EBITDA (A − B)": out["pnl"]["ebitda"] = val
        if label == "Tax": out["pnl"]["tax"] = val
        if label == "PROFIT AFTER TAX (PAT)": out["pnl"]["pat"] = val
        if section == "E" and label[:3].strip().rstrip(".").isdigit():
            out["metrics"][label] = val
    return out


### Cell 5 – Run the whole model in one call and see the P&L

In [ ]:
import importlib, numpy as np, pandas as pd
import engine.inputs, engine.timeline, engine.curves, engine.collections, engine.costs, engine.cashflow, engine.excel_check
for m in (engine.inputs, engine.timeline, engine.curves, engine.collections, engine.costs, engine.cashflow, engine.excel_check):
    importlib.reload(m)
from engine.inputs import *
from engine.cashflow import run_model

project = sample_project()
res = run_model(project)
tl, cv, co, cr, cf = res['timeline'], res['curves'], res['collections'], res['costs'], res['cashflow']
print('✔ ALL CHECKS PASSED' if not res['problems'] else res['problems'])

pnl = cf.pnl
sa = cr.total_sa
statement = pd.DataFrame([
    ['Total revenue (A)', pnl['revenue']], ['Total cost (B)', pnl['total_cost']], ['EBITDA (A − B)', pnl['ebitda']],
    [f"Tax @ {pnl['tax_rate']:.1%} ({project.info.entity})", pnl['tax']], ['PROFIT AFTER TAX (PAT)', pnl['pat']]],
    columns=['P&L', '₹ Crore'])
statement['₹ / sq ft'] = statement['₹ Crore'] * CRORE / sa
statement['% of revenue'] = (statement['₹ Crore'] / pnl['revenue']).map('{:.1%}'.format)
print('\nPROFIT & LOSS'); display(statement.round(2))
print(f"\nJV SPLIT: developer {cr.dev_share:.0%} = ₹{pnl['developer_revenue']:.2f} Cr  |  landowner {1-cr.dev_share:.0%} = ₹{pnl['landowner_revenue']:.2f} Cr"
      f"  |  EBITDA margin on developer's share {pnl['ebitda_margin_on_dev_share']:.1%}")

### Cell 6 – Revenue statement and cost statement (like OUTPUT sections A and B)

In [ ]:
sc = project.scenario()
rev = pd.DataFrame([{'Revenue head': p.name, 'Category': p.category, '₹ Crore': cv.revenue[p.name],
                     '₹ / sq ft': p.rate * (1 + sc.price_change)} for p in project.active_products()])
rev['% of total'] = (rev['₹ Crore'] / rev['₹ Crore'].sum()).map('{:.1%}'.format)
print('A. REVENUE STATEMENT'); display(rev.round(2))

cost = pd.DataFrame([{'Cost head': c.name, 'Applicable': 'Yes' if c.applicable else 'No', '₹ Crore': cr.amounts[c.key],
                      '₹ / sq ft (SA)': cr.amounts[c.key] * CRORE / sa, '% of GDV': cr.amounts[c.key] / cr.total_revenue}
                     for c in project.costs])
cost.loc[len(cost)] = ['TOTAL COST', '', cr.total_cost, cr.total_cost * CRORE / sa, cr.total_cost / cr.total_revenue]
cost['% of GDV'] = cost['% of GDV'].map('{:.2%}'.format)
print('\nB. COST STATEMENT'); display(cost.round(2))

### Cell 7 – Calculators: BAR, AIF and initial investment

In [ ]:
b = cr.bar
print(f"BAR / PREMIUM FAR ({project.bar.authority}, {project.bar.land_use}): free {b['free_sqm']:,.0f} sq m | planned {b['planned_sqm']:,.0f} sq m | "
      f"extra {b['extra_sqm']:,.0f} sq m × ₹{project.bar.reserved_price_sqm:,.0f} × {b['pct']:.0%} = ₹{b['cost']:.2f} Cr")
a = cr.aif
print(f"AIF (if switched on): principal ₹{a['principal']:.2f} Cr at {project.aif.rate:.0%} for {project.aif.tenure_years} yrs → interest ₹{a['cost']:.2f} Cr")
print('\nINITIAL INVESTMENT – JV (paid early, at these quarters)')
ii = pd.DataFrame([{'Item': k, '₹ Crore': v, 'Payout quarter': f"Q{project.initial.payout_q[k]}"} for k, v in cr.initial_items.items()])
display(ii.round(2))
print(f"Total JV ₹{cr.initial_jv:.2f} Cr | Outright would be ₹{cr.initial_outright:.2f} Cr | used for ROCE/CAGR: ₹{cr.initial_used:.2f} Cr")

### Cell 8 – Quarterly cash flow (₹ Cr) – like the Excel CASHFLOW sheet

In [ ]:
Q = [f'Q{i}' for i in range(1, 23)]
rows = {'Customer collections': cf.inflow}
for c in project.costs:
    if abs(cf.outflows[c.key]).sum() > 1e-9: rows[f'  − {c.name}'] = 0.0 - cf.outflows[c.key]
rows.update({'Total costs paid': 0.0 - cf.total_outflow, 'Project net cash flow': cf.project_cf,
             'Advance to landowner': 0.0 - cf.lo_advance, 'Cash paid to landowner (incl. advance)': 0.0 - cf.lo_paid,
             'DEVELOPER NET CASH FLOW': cf.developer_cf, 'Developer – cumulative': np.cumsum(cf.developer_cf)})
display(pd.DataFrame({k: v[:22] for k, v in rows.items()}, index=Q).T.round(2))

### Cell 9 – Key investment metrics (like OUTPUT section E)

In [ ]:
m, bm = cf.metrics, project.finance.benchmarks
qlab = lambda q: f'Q{q}' if q else 'never'
pct = lambda v: 'n/a' if v is None else f'{v:.1%}'
table = [
    ['Project IRR (before landowner share)', pct(m['project_irr']), f"{bm['project_irr']:.0%}", m['status']['project_irr']],
    ['Developer IRR (pre-tax)', pct(m['developer_irr']), f"{bm['developer_irr']:.0%}", m['status']['developer_irr']],
    ['Developer IRR (post-tax, indicative)', pct(m['developer_irr_post_tax']), '', ''],
    ['Equity multiple', f"{m['equity_multiple']:.2f}x", f"{bm['equity_multiple']:.1f}x", m['status']['equity_multiple']],
    ['ROCE (annualised)', pct(m['roce']), f"{bm['roce']:.0%}", m['status']['roce']],
    ['CAGR', pct(m['cagr']), f"{bm['cagr']:.0%}", m['status']['cagr']],
    ['PAT margin', pct(m['pat_margin']), f"{bm['pat_margin']:.0%}", m['status']['pat_margin']],
    ['EBITDA margin', pct(m['ebitda_margin']), f"{bm['ebitda_margin']:.0%}", m['status']['ebitda_margin']],
    ['Developer NPV @ cost of equity', f"₹{m['developer_npv']:.2f} Cr", '', ''],
    ['Project NPV @ cost of equity', f"₹{m['project_npv']:.2f} Cr", '', ''],
    ['Peak investment (developer)', f"₹{m['peak_investment']:.2f} Cr in {qlab(m['peak_quarter'])}" + (f" ({m['peak_date']:%b %Y})" if m['peak_date'] else ''), '', ''],
    ['Largest single-quarter deficit', f"₹{m['largest_quarter_deficit']:.2f} Cr", '', ''],
    ['Break-even quarter – project / developer', f"{qlab(m['breakeven_project_q'])} / {qlab(m['breakeven_developer_q'])}", '', ''],
    ['Payback (cumulative cash back to zero)', qlab(m['payback_q']) + (f" ({m['payback_date']:%b %Y})" if m['payback_date'] else ''), '', ''],
    ['Cash multiple (developer)', f"{m['cash_multiple']:.2f}x", '', ''],
    ['Land cost to GDV', pct(m['land_cost_to_gdv']), '', ''],
    ['Initial investment to GDV', pct(m['initial_investment_to_gdv']), '', ''],
]
display(pd.DataFrame(table, columns=['Metric', 'Value', 'Benchmark', 'Status']))

### Cell 10 – Developer cash flow chart and yearly (financial-year) summary

In [ ]:
import plotly.graph_objects as go
Q32 = [f'Q{i}' for i in range(1, 33)]
fig = go.Figure()
fig.add_bar(x=Q32, y=cf.developer_cf[:32], name='Net cash flow (quarter)',
            marker_color=['#C00000' if v < 0 else '#5B9BD5' for v in cf.developer_cf[:32]])
fig.add_scatter(x=Q32, y=np.cumsum(cf.developer_cf)[:32], name='Cumulative (funding curve)', line=dict(color='#ED7D31', width=3))
fig.update_layout(title=f"Developer cash flow (₹ Cr) – peak investment ₹{m['peak_investment']:.1f} Cr, payback {qlab(m['payback_q'])}", height=420)
fig.show()
fy = pd.DataFrame(cf.fy_summary).T
fy = fy[(fy.abs().sum(axis=1)) > 1e-9]
fy['Developer cumulative'] = fy['developer_net'].cumsum()
display(fy.rename(columns={'collections': 'Collections', 'costs_paid': 'Costs paid', 'paid_to_landowner': 'Paid to landowner',
                           'developer_net': 'Developer net'}).round(2))

### Cell 11 – Compare with the Excel FINAL file ✅
Checks every cost amount, every quarterly cash row, the P&L and all 18 key metrics.

In [ ]:
from engine.excel_check import read_excel_cashflow
xl_path = 'data/Feasibility_Engine_FINAL.xlsx'
if not os.path.exists(xl_path):
    try:
        from google.colab import files
        print('Please choose Feasibility_Engine_FINAL.xlsx …'); up = files.upload(); os.replace(next(iter(up)), xl_path)
    except Exception as e:
        print('Upload skipped:', e)
if os.path.exists(xl_path):
    x = read_excel_cashflow(xl_path)
    d_cost = max(abs(a - b) for a, b in zip(cr.amounts.values(), x['costs']))
    d_out = max(abs(o - xo).max() for o, xo in zip(cf.outflows.values(), x['outflows']))
    d_rows = max(abs(getattr(cf, k) - v).max() for k, v in x['rows'].items())
    d_pnl = max(abs(cf.pnl[k] - v) for k, v in x['pnl'].items())
    for lab, d in [('Cost amounts (21 heads)', d_cost), ('Outflows per cost head per quarter', d_out),
                   ('Collections / landowner / project & developer cash rows', d_rows), ('P&L (revenue, cost, EBITDA, tax, PAT)', d_pnl)]:
        print(('✔' if d < 1e-9 else '✖'), f'{lab:58} largest difference {d:.1e}')
    keys = ['project_irr', 'developer_irr', 'equity_multiple', 'roce', 'cagr', 'pat_margin', 'ebitda_margin', 'peak_investment',
            'land_cost_to_gdv', 'initial_investment_to_gdv', 'breakeven_project_q', 'breakeven_developer_q', 'payback_q',
            'project_npv', 'developer_npv', 'developer_irr_post_tax', 'largest_quarter_deficit', 'cash_multiple']
    ok = 0
    for (lab, xv), k in zip(x['metrics'].items(), keys):
        pv = cf.metrics[k]
        same = (xv.startswith('Q') and pv == int(xv[1:])) if isinstance(xv, str) else (pv is not None and abs(pv - xv) < 1e-6)
        ok += same
        if not same: print('✖', lab, 'Excel', xv, 'Python', pv)
    print(('✔' if ok == len(keys) else '✖'), f'Key metrics identical: {ok} of {len(keys)}')
else:
    print('Excel file not available – skipped.')

### Cell 12 – One engine, many what-ifs (each line is a full model run)

In [ ]:
import copy
def line(p, label):
    r = run_model(p); k = r['cashflow'].metrics; pl = r['cashflow'].pnl
    ok = '✔' if not r['problems'] else '✖'
    return [ok, label, round(pl['revenue'], 1), round(pl['ebitda'], 1), pct(k['developer_irr']), round(k['developer_npv'], 1),
            round(k['peak_investment'], 1), qlab(k['payback_q'])]
tests = [(project, 'Base case')]
p = copy.deepcopy(project); p.info.scenario = 'Worst Case'; tests.append((p, 'Worst Case'))
p = copy.deepcopy(project); p.info.scenario = 'Best Case'; tests.append((p, 'Best Case'))
p = copy.deepcopy(project); p.info.deal_type = 'Outright'; tests.append((p, 'Outright deal (land paid upfront)'))
p = copy.deepcopy(project); p.info.project_type = 'Apartments'; tests.append((p, 'Apartments-only project'))
p = copy.deepcopy(project); p.products[1].construction_duration_q = 24; tests.append((p, 'Apartment construction 16 → 24 qtrs'))
p = copy.deepcopy(project); p.products[1].clp_plan = 'CLP – 17-Floor Tower'; tests.append((p, 'Apartments on 17-floor plan'))
p = copy.deepcopy(project); p.catch_up_lag = 0; tests.append((p, 'Catch-up paid at booking'))
display(pd.DataFrame([line(p, l) for p, l in tests],
        columns=['', 'What-if', 'GDV ₹Cr', 'EBITDA ₹Cr', 'Developer IRR', 'Dev NPV ₹Cr', 'Peak ₹Cr', 'Payback']))

---
### ✅ Step 4 complete – the full calculation engine now runs in Python
`engine/` holds `inputs.py`, `timeline.py`, `curves.py`, `collections.py`, `costs.py`, `cashflow.py` and `excel_check.py`.
`run_model(project)` reproduces the entire Excel workbook in a fraction of a second.

**Next – Step 5:** an automatic test suite (one click re-runs every Excel comparison and scenario check), so any future change to the code is instantly verified. After that we start building the app screens.